# 1단계 모델 검증: 전체 데이터 랜덤 분할 (8:2)

## 1. 라이브러리 임포트

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

## 2. 데이터 불러오기 및 전처리

In [ ]:
# 검단 포함 전체 신도시 데이터 불러오기
folder_path = r"C:\Users\SJ\Desktop\26-1\데이터마이닝\파주시아파트"
df = pd.read_csv(folder_path + r"\new_city2.csv", encoding='utf-8-sig')
df = df[df['도시명'].isin(['운정', '광교', '판교', '검단'])]

# 거래금액 전처리 (쉼표 제거 후 정수 변환)
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)

# m2당가격 파생변수 생성
df['m2당가격'] = df['거래금액(만원)'] / df['전용면적(㎡)']

print(f"전체 데이터: {len(df)}건")

## 3. 이상치 제거

In [ ]:
# 전용면적 33제곱미터 미만 제거
df = df[df['전용면적(㎡)'] >= 33]

# 발표후경과년수 3년미만 필터링
before = len(df)
df = df[df['발표후경과년수'] >= 3]
print(f"발표후경과년수 3 미만 제거: {before - len(df)}개 → 남은 데이터: {len(df)}개")

# m2당가격 z-score 기준 이상치 제거 (|z| > 2인 데이터 제거)
mean = df['m2당가격'].mean()
std = df['m2당가격'].std()
z_scores = (df['m2당가격'] - mean) / std
df = df[z_scores.abs() <= 2]

print(f"이상치 제거 후: {len(df)}건")

## 4. 독립변수 설정 및 훈련/테스트 분할

In [ ]:
# 독립변수 목록
features = ['건축년도', '층',
            '지하철호선개수', '기차역까지의거리',
            '가장 가까운 지하철역까지의 거리', '가장 가까운 IC와의 거리',
            '발표후경과년수', 'CPI', '계약연도', '서울도심거리',
            '단지별_세대수', '도시별_세대수']

# 결측치 제거
df = df.dropna(subset=features + ['m2당가격'])

# 독립변수와 타겟 분리
X = df[features]
y = df['m2당가격']

# 훈련/테스트 분할 (8:2)
train_input, test_input, train_target, test_target = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"훈련 세트: {train_input.shape}")
print(f"테스트 세트: {test_input.shape}")

## 5. 표준화

In [ ]:
# 표준화
ss = StandardScaler()
ss.fit(train_input)
train_scaled = ss.transform(train_input)
test_scaled = ss.transform(test_input)

## 6. 최적 alpha 탐색

In [ ]:
# alpha 값별로 Lasso 모델 학습 및 점수 저장
train_score = []
test_score = []
alpha_list = [0.001, 0.01, 0.1, 1, 10, 100]

for alpha in alpha_list:
    lasso_a = Lasso(alpha=alpha, max_iter=1000000)
    lasso_a.fit(train_scaled, train_target)
    train_score.append(lasso_a.score(train_scaled, train_target))
    test_score.append(lasso_a.score(test_scaled, test_target))

# 테스트 점수가 가장 높은 alpha 선택
best_idx = test_score.index(max(test_score))
best_alpha = alpha_list[best_idx]
print(f"최적 alpha: {best_alpha}")

## 7. 최적 alpha로 최종 모델 학습

In [ ]:
# 최적 alpha로 Lasso 모델 학습
lasso_best = Lasso(alpha=best_alpha, max_iter=1000000)
lasso_best.fit(train_scaled, train_target)

print(f"{'='*50}")
print(f"1단계 모델 성능 (랜덤 8:2 분할)")
print(f"{'='*50}")
print(f"최적 alpha: {best_alpha}")
print(f"훈련 R²: {lasso_best.score(train_scaled, train_target):.4f}")
print(f"테스트 R²: {lasso_best.score(test_scaled, test_target):.4f}")